# 🔐 Notebook 2: Presigned URLs

Give clients temporary, scoped credentials to upload/download directly from storage.

## Learning Objectives

By the end of this notebook, you'll understand:
- How presigned URLs work
- Generating upload and download URLs
- Adding security constraints
- URL expiration and validation

In [ ]:
import boto3
from botocore.config import Config
import requests
import uuid
from datetime import datetime

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

BUCKET = 'uploads'

print("✅ Connected to MinIO!")
print("📊 Open MinIO Console: http://localhost:9001")
print("   Login: minioadmin / minioadmin")

## 🔐 How Presigned URLs Work

In [ ]:
print("🔐 How Presigned URLs Work")
print("=" * 60)
print("""
A presigned URL is like a VIP pass with an expiration.

ANATOMY OF A PRESIGNED URL:
─────────────────────────────────────────────────────────────

https://mybucket.s3.amazonaws.com/uploads/user123/video.mp4
  ?X-Amz-Algorithm=AWS4-HMAC-SHA256
  &X-Amz-Credential=AKIAEXAMPLE/20240115/us-east-1/s3/aws4_request
  &X-Amz-Date=20240115T000000Z
  &X-Amz-Expires=900           ← Valid for 15 minutes
  &X-Amz-SignedHeaders=host
  &X-Amz-Signature=abc123...   ← Cryptographic signature

HOW IT WORKS:
─────────────────────────────────────────────────────────────

1. Your server creates signature using SECRET key
2. Signature encodes: bucket, key, expiry, permissions
3. Client gets URL (never sees your secret)
4. Storage validates signature using same secret
5. If valid + not expired → allow access

SECURITY:
• Anyone with URL can use it (until expiry)
• URL is tied to specific object and operation
• Short expiry = smaller attack window
""")

## 📤 Generate Upload URL

In [ ]:
def generate_upload_url(user_id: str, filename: str, expires_in: int = 3600) -> dict:
    file_id = str(uuid.uuid4())
    storage_key = f"{user_id}/{file_id}/{filename}"
    
    presigned_url = s3.generate_presigned_url(
        'put_object',
        Params={
            'Bucket': BUCKET,
            'Key': storage_key,
        },
        ExpiresIn=expires_in
    )
    
    return {
        'file_id': file_id,
        'storage_key': storage_key,
        'upload_url': presigned_url,
        'expires_in': expires_in
    }

print("📤 Generating Upload URL")
print("=" * 60)

upload_info = generate_upload_url('user123', 'vacation.jpg')

print(f"\n📋 Upload Info:")
print(f"   File ID: {upload_info['file_id']}")
print(f"   Storage Key: {upload_info['storage_key']}")
print(f"   Expires In: {upload_info['expires_in']} seconds")
print(f"\n🔗 Upload URL (truncated):")
print(f"   {upload_info['upload_url'][:80]}...")

In [ ]:
print("📤 Uploading File Using Presigned URL")
print("=" * 60)

file_content = b"Hello, this is a test file for presigned URL upload!"

response = requests.put(
    upload_info['upload_url'],
    data=file_content,
    headers={'Content-Type': 'image/jpeg'}
)

print(f"\n📊 Upload Result:")
print(f"   Status: {response.status_code}")
print(f"   Success: {'✅ Yes' if response.status_code == 200 else '❌ No'}")

if response.status_code == 200:
    print(f"\n🎉 File uploaded directly to MinIO!")
    print(f"   Your server never touched the bytes!")
    print(f"\n📊 Check MinIO Console: http://localhost:9001")
    print(f"   Bucket: {BUCKET}")
    print(f"   Key: {upload_info['storage_key']}")

## 📥 Generate Download URL

In [ ]:
def generate_download_url(storage_key: str, expires_in: int = 3600) -> str:
    return s3.generate_presigned_url(
        'get_object',
        Params={
            'Bucket': BUCKET,
            'Key': storage_key,
        },
        ExpiresIn=expires_in
    )

print("📥 Generating Download URL")
print("=" * 60)

download_url = generate_download_url(upload_info['storage_key'])

print(f"\n🔗 Download URL (truncated):")
print(f"   {download_url[:80]}...")

response = requests.get(download_url)
print(f"\n📊 Download Result:")
print(f"   Status: {response.status_code}")
print(f"   Content: {response.content.decode()[:50]}...")
print("\n✅ Downloaded directly from MinIO!")

## 🔒 Adding Security Constraints

In [ ]:
print("🔒 Security Constraints")
print("=" * 60)
print("""
Presigned URLs can include CONDITIONS that must be met.

COMMON CONSTRAINTS:
─────────────────────────────────────────────────────────────

1. CONTENT LENGTH RANGE
   • Prevent 10GB upload when expecting 10MB
   • ["content-length-range", 0, 10485760]  (max 10MB)

2. CONTENT TYPE
   • Ensure profile pic is image, not video
   • {"Content-Type": "image/jpeg"}

3. EXPIRATION TIME
   • Short for sensitive ops (5-15 min)
   • Longer for large uploads (1-24 hours)

4. KEY PREFIX
   • User can only upload to their folder
   • ["starts-with", "$key", "users/user123/"]
""")

In [ ]:
def generate_constrained_upload(user_id: str, filename: str, max_size_mb: int = 10) -> dict:
    file_id = str(uuid.uuid4())
    storage_key = f"{user_id}/{file_id}/{filename}"
    
    conditions = [
        ["content-length-range", 0, max_size_mb * 1024 * 1024],
        {"bucket": BUCKET},
        ["starts-with", "$key", f"{user_id}/"],
    ]
    
    fields = {
        "key": storage_key,
    }
    
    presigned_post = s3.generate_presigned_post(
        Bucket=BUCKET,
        Key=storage_key,
        Conditions=conditions,
        ExpiresIn=3600
    )
    
    return {
        'file_id': file_id,
        'storage_key': storage_key,
        'post_url': presigned_post['url'],
        'fields': presigned_post['fields'],
        'max_size_mb': max_size_mb
    }

print("🔒 Generating Constrained Upload URL")
print("=" * 60)

constrained = generate_constrained_upload('user456', 'profile.jpg', max_size_mb=5)

print(f"\n📋 Constrained Upload Info:")
print(f"   File ID: {constrained['file_id']}")
print(f"   Max Size: {constrained['max_size_mb']}MB")
print(f"   POST URL: {constrained['post_url']}")
print(f"\n🔐 Fields to include:")
for key, value in constrained['fields'].items():
    display_value = value[:50] + '...' if len(str(value)) > 50 else value
    print(f"   {key}: {display_value}")

In [ ]:
print("📤 Upload Using POST with Constraints")
print("=" * 60)

files = {'file': ('profile.jpg', b'Small valid file content', 'image/jpeg')}

response = requests.post(
    constrained['post_url'],
    data=constrained['fields'],
    files=files
)

print(f"\n📊 Upload Result:")
print(f"   Status: {response.status_code}")
print(f"   Success: {'✅ Yes' if response.status_code == 204 else '❌ No'}")

## ⏰ URL Expiration Demo

In [ ]:
print("⏰ URL Expiration Demo")
print("=" * 60)

short_lived_url = s3.generate_presigned_url(
    'get_object',
    Params={'Bucket': BUCKET, 'Key': upload_info['storage_key']},
    ExpiresIn=5
)

print("\n🔗 Generated URL that expires in 5 seconds...")
print(f"   URL: {short_lived_url[:60]}...")

print("\n⏱️ Testing immediately...")
response = requests.get(short_lived_url)
print(f"   Status: {response.status_code} {'✅' if response.status_code == 200 else '❌'}")

import time
print("\n⏳ Waiting 6 seconds for URL to expire...")
time.sleep(6)

print("\n⏱️ Testing after expiration...")
response = requests.get(short_lived_url)
print(f"   Status: {response.status_code} {'✅' if response.status_code == 200 else '❌ Expired!'}")

print("\n✅ Expired URLs are automatically rejected!")

## 🧪 Quick Quiz

1. **What happens if someone intercepts a presigned URL?**

2. **Why use POST with conditions instead of PUT?**

3. **What expiration time would you use for a video upload vs profile pic?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. If URL is intercepted:")
print("   - They can use it until it expires")
print("   - Mitigation: Short expiry (5-15 min)")
print("   - URL is tied to specific operation")
print()
print("2. POST vs PUT:")
print("   - POST allows server-side conditions")
print("   - Validate size, content-type at storage")
print("   - PUT is simpler but less restrictive")
print()
print("3. Expiration times:")
print("   - Profile pic: 5-15 min (small, quick)")
print("   - Video upload: 1-24 hours (large, slow)")
print("   - Balance: security vs user experience")

## 📚 Summary

### Key Takeaways

1. **Presigned URLs** - Temporary credentials baked into URL
2. **No server secret exposed** - Client never sees your credentials
3. **Constraints** - Enforce size, type at storage level
4. **Expiration** - Always set reasonable expiry
5. **PUT vs POST** - POST for constraints, PUT for simplicity

### Next Up

In **Notebook 3**, we'll learn about resumable uploads:
- Multipart upload API
- Handling failures mid-upload
- Progress tracking